In [18]:
# 코리안 블라썸 모델 
import gradio as gr
import os
import torch
from transformers import AutoProcessor, MllamaForConditionalGeneration
from PIL import Image

In [19]:
# Determine the device (GPU if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [20]:
# Load the model and processor
model_name = """Bllossom/llama-3.2-Korean-Bllossom-AICA-5B"""
model = MllamaForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map='cpu'
)

Loading checkpoint shards: 100%|██████████| 3/3 [00:21<00:00,  7.26s/it]


In [21]:
# Move the model to the appropriate device (GPU if available)
model.to(device)
processor = AutoProcessor.from_pretrained(model_name)
# VRAM을 많이 먹을 경우 아래 코드 실행
model.eval()

MllamaForConditionalGeneration(
  (vision_model): MllamaVisionModel(
    (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), padding=valid, bias=False)
    (gated_positional_embedding): MllamaPrecomputedPositionEmbedding(
      (tile_embedding): Embedding(9, 8197120)
    )
    (pre_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
      (embedding): Embedding(9, 5120)
    )
    (post_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
      (embedding): Embedding(9, 5120)
    )
    (layernorm_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (layernorm_post): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): MllamaVisionEncoder(
      (layers): ModuleList(
        (0-31): 32 x MllamaVisionEncoderLayer(
          (self_attn): MllamaVisionSdpaAttention(
            (q_proj): Linear(in_features=1280, out_features=1280, bias=False)
            (k_proj): Linear(in_features=1280, out_features=1280, b

In [24]:
torch.cuda.empty_cache()
with torch.no_grad():
    # LLM으로 사용할 때
    # messages = [
    # {'role':'user', 'content': [
    #     {'type':'text', 'text':'자음과 모음을 순서대로 결합해서 자연스러운 한국어 문장으로 만들어줘 \n\n ㅂ ㅏ ㄴ ㄱ ㅏ ㅇ ㅜ ㅓ'}
    #     ]},
    # ]
    messages = [
    {'role':'user', 'content': [
        {'type':'text', 'text':'한국어를 일본어로 바꿔줘 \n\n 현관문을 열어줘'}
        ]},
    ]
    inputs = processor.tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True,
        return_tensors='pt').to(model.device)
    outputs = model.generate(
        inputs, 
        max_new_tokens=512,
        temperature=0.1,
        eos_token_id=processor.tokenizer.convert_tokens_to_ids('<|eot_id|>'))

# # Generate a response from the model
#     with torch.cuda.amp.autocast():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=512,
#             use_cache=True,
#             temperature=0.1,
#             eos_token_id=processor.tokenizer.convert_tokens_to_ids('<|eot_id|>'),
#             )

    # Decode the output to return the final response
    response = processor.decode(outputs[0])
    response = response[
        response.rindex('<|start_header_id|>assistant<|end_header_id|>\n\n')+\
        len('<|start_header_id|>assistant<|end_header_id|>\n\n'):].replace('<|eot_id|>','')

    print(response)

ドアを開けろ。
